In [2]:
# --- Preparación y datos ---
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, VectorAssembler
from pyspark.ml.regression import GeneralizedLinearRegression

spark = SparkSession.builder.appName("04_model").getOrCreate()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")   # acelera toPandas

BUCKET = "taxis-chicago-grandes-datos-506301"
GOLD = f"gs://{BUCKET}/gold"
MODELS = f"gs://{BUCKET}/models"
EXPORT = f"{GOLD}/app_export"
START, CUTOFF = "2022-01-01", "2023-07-01"

FEATURES = ["zone", "hour", "day_of_week", "month", "is_weekend", "is_holiday",
            "temp_c", "prcp_mm", "snow", "profile_4w"]

data = (spark.read.parquet(f"{GOLD}/zone_hour_demand")
        .filter(F.col("date") >= START)
        .select("date", *FEATURES, F.col("trips").cast("double").alias("trips"))
        .na.drop()
        .cache())
train = data.filter(F.col("date") < CUTOFF)
test = data.filter(F.col("date") >= CUTOFF)
print(f"train: {train.count():,}  test: {test.count():,}")

26/10/05 18:11:15 WARN CacheManager: Asked to cache already cached data.


train: 1,009,008  test: 340,032


In [3]:
# --- modelo base, regresión de Poisson en Spark ML (distribuido) ---
CATEGORICAL = {"zone": 1, "hour": 0, "day_of_week": 1, "month": 1}   # columna: su valor mínimo
NUMERIC = ["is_holiday", "temp_c", "prcp_mm", "snow", "log_profile"]
# is_weekend no entra: ya está contenido en day_of_week


def prepare(df):
    for column, first in CATEGORICAL.items():
        df = df.withColumn(f"{column}_idx", F.col(column) - first)       # índices desde 0
    return df.withColumn("log_profile", F.log1p("profile_4w"))


encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in CATEGORICAL],
                        outputCols=[f"{c}_ohe" for c in CATEGORICAL], dropLast=True)
assembler = VectorAssembler(inputCols=[f"{c}_ohe" for c in CATEGORICAL] + NUMERIC,
                            outputCol="features")
glm = GeneralizedLinearRegression(family="poisson", link="log", labelCol="trips",
                                  featuresCol="features", maxIter=25, regParam=0.001)
poisson = Pipeline(stages=[encoder, assembler, glm]).fit(prepare(train))
poisson.write().overwrite().save(f"{MODELS}/demand_poisson_v1")


In [4]:
# --- exportar los pesos de Poisson a JSON, con el nombre de cada variable ---
sizes = poisson.stages[0].categorySizes
names = []
for (column, first), size in zip(CATEGORICAL.items(), sizes):
    names += [f"{column}_{i + first}" for i in range(size - 1)]   # dropLast: la última categoría es la base
names += NUMERIC

glm_model = poisson.stages[-1]
coefficients = glm_model.coefficients.toArray()
assert len(names) == len(coefficients), "los nombres no cuadran con los coeficientes"
poisson_json = {"intercept": float(glm_model.intercept),
                "coefficients": {n: float(c) for n, c in zip(names, coefficients)}}
with open("/tmp/demand_poisson_v1.json", "w") as f:
    json.dump(poisson_json, f, indent=1)
print(f"Coeficientes exportados: {len(names)}")


def predict_poisson(model, x):
    """La misma cuenta que hará la app: solo numpy, sin Spark."""
    c = model["coefficients"]
    eta = model["intercept"] + sum(
        x[column].map(lambda v, column=column: c.get(f"{column}_{int(v)}", 0.0))
        for column in ["zone", "hour", "day_of_week", "month"])
    eta = eta + sum(c[n] * x[n] for n in ["is_holiday", "temp_c", "prcp_mm", "snow"])
    eta = eta + c["log_profile"] * np.log1p(x["profile_4w"])
    return np.exp(eta)


Coeficientes exportados: 121


In [5]:
# ---llevar train y test a pandas (ya son tablas pequeñas) ---
train_pd = train.toPandas()
test_pd = (poisson.transform(prepare(test))
           .select("date", *FEATURES, "trips", F.col("prediction").alias("poisson_spark"))
           .toPandas())
test_pd["poisson"] = predict_poisson(poisson_json, test_pd)
gap = (test_pd["poisson"] - test_pd["poisson_spark"]).abs().max()
print(f"Diferencia máxima entre Spark y el JSON exportado: {gap:.6f} viajes")

Diferencia máxima entre Spark y el JSON exportado: 0.000000 viajes


In [7]:
# --- modelo avanzado, XGBoost en el nodo maestro ---
subprocess.run(f"gsutil -q cp gs://{BUCKET}/libs/xgboost*.whl /tmp/", shell=True, check=True)
subprocess.run(f"{sys.executable} -m pip install --quiet --upgrade --no-deps --target /tmp/xgb_lib /tmp/xgboost*.whl",
               shell=True, check=True)
sys.path.insert(0, "/tmp/xgb_lib")
import xgboost as xgb
print("XGBoost", xgb.__version__)

dtrain = xgb.DMatrix(train_pd[FEATURES], label=train_pd["trips"])
dtest = xgb.DMatrix(test_pd[FEATURES])
params = {"objective": "count:poisson", "tree_method": "hist", "max_depth": 6, "eta": 0.1,
          "subsample": 0.8, "seed": 42}
booster = xgb.train(params, dtrain, num_boost_round=300)
booster.save_model("/tmp/demand_xgboost_v1.json")
test_pd["xgboost"] = booster.predict(dtest)

XGBoost 2.1.4


In [ ]:
# --- comparación en el mismo periodo de prueba ---
top_zones = train_pd.groupby("zone")["trips"].sum().nlargest(10).index
is_top = test_pd["zone"].isin(top_zones)


def evaluate(column):
    error = test_pd["trips"] - test_pd[column]
    return {"mae": error.abs().mean(),
            "rmse": np.sqrt((error ** 2).mean()),
            "r2": 1 - (error ** 2).sum() / ((test_pd["trips"] - test_pd["trips"].mean()) ** 2).sum(),
            "mae_top10_zones": error[is_top].abs().mean(),
            "mae_other_zones": error[~is_top].abs().mean()}


comparison = pd.DataFrame({"profile_4w": evaluate("profile_4w"),
                           "poisson": evaluate("poisson"),
                           "xgboost": evaluate("xgboost")}).T.round(4)
comparison.index.name = "model"
print(comparison)
print(f"\nMenor MAE en prueba: {comparison['mae'].idxmin()}")

               mae     rmse      r2  mae_top10_zones  mae_other_zones
model                                                                
profile_4w  2.4502  10.0088  0.9242          13.5934           0.7870
poisson     2.4429   9.3090  0.9345          13.0841           0.8547
xgboost     2.4497   9.4937  0.9318          13.5577           0.7917

Menor MAE en prueba: poisson


In [ ]:
# --- qué aprendió cada modelo ---
effects = {n: f"{(np.exp(poisson_json['coefficients'][n]) - 1) * 100:+.1f}%"
           for n in ["is_holiday", "snow", "prcp_mm", "temp_c"]}
print("Poisson, cambio en la demanda por cada unidad de la variable:", effects)

gain = pd.Series(booster.get_score(importance_type="gain"))
print("\nXGBoost, importancia de cada variable (% de la ganancia):")
print((gain / gain.sum() * 100).round(1).sort_values(ascending=False))

# --- Celda 8: guardar modelos y exportar lo que necesita la app ---
subprocess.run(f"gsutil -q cp /tmp/demand_poisson_v1.json /tmp/demand_xgboost_v1.json {MODELS}/",
               shell=True, check=True)


def export(df, name):
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{EXPORT}/{name}")


export(spark.createDataFrame(comparison.reset_index()), "model_metrics")

last_day = data.agg(F.max("date")).first()[0]
export(data.filter(F.col("date") > F.date_sub(F.lit(last_day), 28))
       .groupBy("zone", "day_of_week", "hour")
       .agg(F.round(F.avg("trips"), 2).alias("profile_4w")),
       "recent_profile")
export(spark.read.parquet(f"{GOLD}/dim_zones"), "dim_zones")

Poisson, cambio en la demanda por cada unidad de la variable: {'is_holiday': '-37.1%', 'snow': '+1.5%', 'prcp_mm': '+0.1%', 'temp_c': '+0.1%'}

XGBoost, importancia de cada variable (% de la ganancia):
profile_4w     97.7
is_holiday      0.5
month           0.3
temp_c          0.3
prcp_mm         0.3
snow            0.3
zone            0.2
day_of_week     0.2
hour            0.1
is_weekend      0.1
dtype: float64
